In [ ]:
!pip install -q ctranslate2==4.5.0 transformers==4.44.0 huggingface_hub tokenizers torch

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import subprocess
import os

MODEL_ID = "momosl/whisper-wolof-v1"
OUTPUT_DIR = "/content/whisper-wolof-ct2"

print(f"Conversion de {MODEL_ID} vers CTranslate2 (int8)...")
print("(~5 min)\n")

result = subprocess.run([
    "ct2-whisper-converter",
    "--model", MODEL_ID,
    "--output_dir", OUTPUT_DIR,
    "--quantization", "int8",
    "--copy_files", "tokenizer.json", "preprocessor_config.json",
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("ERREUR:", result.stderr)
else:
    print(f"\nConversion terminee ! Fichiers:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
        print(f"  {f} ({size:.1f} Mo)")

In [ ]:
from huggingface_hub import HfApi

HF_REPO_CT2 = "momosl/whisper-wolof-v1-ct2"

api = HfApi()
api.create_repo(HF_REPO_CT2, exist_ok=True)

print(f"Push vers {HF_REPO_CT2}...")
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=HF_REPO_CT2,
    commit_message="Whisper Wolof v1 - CTranslate2 int8"
)

print(f"\n{'='*60}")
print(f"   MODELE CT2 PUBLIE !")
print(f"   https://huggingface.co/{HF_REPO_CT2}")
print(f"")
print(f"   Ce modele est compatible avec faster-whisper")
print(f"   et sera utilise par la Lambda AWS.")
print(f"{'='*60}")